# Loss Functions

In [6]:
import numpy as np
rng = np.random.default_rng(7)
from typing import Optional

# Import from TinyTorch package (previous modules must be completed and exported)
from tinytorch.core.tensor import Tensor
from tinytorch.core.activations import ReLU
from tinytorch.core.layers import Linear

# Constants for numerical stability
EPSILON = 1e-7  # Small value to prevent log(0) and numerical instability

### - Log-Softmax: The Numerically Stable Foundation

Before implementing loss functions, we need a reliable way to compute log-softmax. This function is the numerically stable backbone of classification losses.

### - Why Log-Softmax Matters

Naive softmax can explode with large numbers

TODO: Implement numerically stable log-softmax using the log-sum-exp trick

APPROACH:
- Find maximum along dimension (for stability)
- Subtract max from input (prevents overflow)
- Compute log(sum(exp(shifted_input)))
- Return input - max - log_sum_exp

In [7]:
def log_softmax(x: Tensor, dim: int = -1) -> Tensor:
    """
    Compute log-softmax with numerical stability.

    HINT: Use np.max(x.data, axis=dim, keepdims=True) to preserve dimensions
    """
    ### BEGIN SOLUTION
    # Step 1: Find max along dimension for numerical stability
    max_vals = np.max(x.data, axis=dim, keepdims=True)

    # Step 2: Subtract max to prevent overflow
    shifted = x.data - max_vals

    # Step 3: Compute log(sum(exp(shifted)))
    log_sum_exp = np.log(np.sum(np.exp(shifted), axis=dim, keepdims=True))

    # Step 4: Return log_softmax = input - max - log_sum_exp
    result = x.data - max_vals - log_sum_exp

    return Tensor(result)

In [8]:
logits = Tensor([[1.0, 2.0, 3.0], [0.1, 0.2, 0.9]])

result = log_softmax(logits, dim=1)
print("Log-Softmax shape:", result.data.shape)

Log-Softmax shape: (2, 3)


### 🧪 Unit Test: Log-Softmax

This test validates our log_softmax function works correctly with numerical stability.

**What we're testing**: Numerical stability and correctness of log-softmax computation

**Why it matters**: Foundation for cross-entropy loss - must handle large values without overflow

**Expected**: Stable results even with extreme inputs, softmax sums to 1

In [9]:
def test_unit_log_softmax():
    """🧪 Test log_softmax numerical stability and correctness."""
    print("🧪 Unit Test: Log-Softmax...")

    # Test basic functionality
    x = Tensor([[1.0, 2.0, 3.0], [0.1, 0.2, 0.9]])
    result = log_softmax(x, dim=-1)

    # Verify shape preservation
    assert result.shape == x.shape, f"Shape mismatch: expected {x.shape}, got {result.shape}"

    # Verify log-softmax properties: exp(log_softmax) should sum to 1
    softmax_result = np.exp(result.data)
    row_sums = np.sum(softmax_result, axis=-1)
    assert np.allclose(row_sums, 1.0, atol=1e-6), f"Softmax doesn't sum to 1: {row_sums}"

    # Test numerical stability with large values
    large_x = Tensor([[100.0, 101.0, 102.0]])
    large_result = log_softmax(large_x, dim=-1)
    assert not np.any(np.isnan(large_result.data)), "NaN values in result with large inputs"
    assert not np.any(np.isinf(large_result.data)), "Inf values in result with large inputs"

    print("✅ log_softmax works correctly with numerical stability!")

if __name__ == "__main__":
    test_unit_log_softmax()

🧪 Unit Test: Log-Softmax...
✅ log_softmax works correctly with numerical stability!


# 1. MSELoss: Measuring Continuous Prediction Quality

Mean Squared Error is the workhorse of regression problems. It measures how far your continuous predictions are from the true values.

### When to Use MSE

**Perfect for:**
- House price prediction ($200k vs $195k)
- Temperature forecasting (25°C vs 23°C)
- Stock price prediction ($150 vs $148)
- Any continuous value where "distance" matters

### Why Square the Errors?

1. **Positive penalties**: (-10)² = 100, same as (+10)² = 100
2. **Heavy punishment for large errors**: Error of 20 → penalty of 400
3. **Smooth gradients**: Quadratic function has nice derivatives for optimization
4. **Statistical foundation**: Maximum likelihood for Gaussian noise

TODO: Implement MSE loss calculation

APPROACH:
1. Compute difference: predictions - targets
2. Square the differences: diff²
3. Take mean across all elements

In [10]:
class MSELoss:
    """Mean Squared Error loss for regression tasks."""

    def __init__(self):
        """Initialize MSE loss function."""
        pass

    def forward(self, predictions: Tensor, targets: Tensor) -> Tensor:
        """
        Compute mean squared error between predictions and targets.

        HINTS:
        - Use (predictions.data - targets.data) for element-wise difference
        - Square with **2 or np.power(diff, 2)
        - Use np.mean() to average over all elements
        """
        ### BEGIN SOLUTION
        # Step 1: Compute element-wise difference
        diff = predictions.data - targets.data

        # Step 2: Square the differences
        squared_diff = diff ** 2

        # Step 3: Take mean across all elements
        mse = np.mean(squared_diff)

        return Tensor(mse)
        ### END SOLUTION

    def __call__(self, predictions: Tensor, targets: Tensor) -> Tensor:
        """Allows the loss function to be called like a function."""
        return self.forward(predictions, targets)

    def backward(self) -> Tensor:
        """
        Compute gradients (placeholder — gradient computation is separate).

        For now, this is a stub that students can ignore.
        """
        pass

In [12]:
loss_fn = MSELoss()
predictions = Tensor([1.0, 2.0, 3.0])
targets = Tensor([1.5, 2.5, 2.8])
loss = loss_fn(predictions, targets)
print(f"MSE Loss: {loss.data:.4f}")

MSE Loss: 0.1800


### 🧪 Unit Test: MSE Loss

This test validates our MSELoss implementation with various prediction scenarios.

**What we're testing**: Mean squared error calculation with perfect and imperfect predictions

**Why it matters**: MSE is the foundation for regression - must be mathematically correct

**Expected**: Zero loss for perfect predictions, positive loss for errors, non-negative always

In [13]:
def test_unit_mse_loss():
    """🧪 Test MSELoss implementation and properties."""
    print("🧪 Unit Test: MSE Loss...")

    loss_fn = MSELoss()

    # Test perfect predictions (loss should be 0)
    predictions = Tensor([1.0, 2.0, 3.0])
    targets = Tensor([1.0, 2.0, 3.0])
    perfect_loss = loss_fn.forward(predictions, targets)
    assert np.allclose(perfect_loss.data, 0.0, atol=EPSILON), f"Perfect predictions should have 0 loss, got {perfect_loss.data}"

    # Test known case
    predictions = Tensor([1.0, 2.0, 3.0])
    targets = Tensor([1.5, 2.5, 2.8])
    loss = loss_fn.forward(predictions, targets)

    # Manual calculation: ((1-1.5)² + (2-2.5)² + (3-2.8)²) / 3 = (0.25 + 0.25 + 0.04) / 3 = 0.18
    expected_loss = (0.25 + 0.25 + 0.04) / 3
    assert np.allclose(loss.data, expected_loss, atol=1e-6), f"Expected {expected_loss}, got {loss.data}"

    # Test that loss is always non-negative
    random_pred = Tensor(rng.standard_normal(10))
    random_target = Tensor(rng.standard_normal(10))
    random_loss = loss_fn.forward(random_pred, random_target)
    assert random_loss.data >= 0, f"MSE loss should be non-negative, got {random_loss.data}"

    print("✅ MSELoss works correctly!")

if __name__ == "__main__":
    test_unit_mse_loss()

🧪 Unit Test: MSE Loss...
✅ MSELoss works correctly!


### CrossEntropyLoss: Measuring Classification Confidence

Cross-entropy loss is the gold standard for multi-class classification. It measures how wrong your probability predictions are and heavily penalizes confident mistakes.

### When to Use Cross-Entropy

**Perfect for:**
- Image classification (cat, dog, bird)
- Text classification (spam, ham, promotion)
- Language modeling (next word prediction)
- Any problem with mutually exclusive classes

### Why Cross-Entropy Works So Well

1. **Probabilistic interpretation**: Measures quality of probability distributions
2. **Strong gradients**: Large penalty for confident mistakes drives fast learning
3. **Smooth optimization**: Log function provides nice gradients
4. **Information theory**: Minimizes "surprise" about correct answers